In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

d:\YouTube\RAG_ML_Chatbot\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def file_loader(path):
  loader = DirectoryLoader(
    path, glob="*.pdf", loader_cls=PyPDFLoader
  )
  documents = loader.load()
  return documents


In [4]:
extracted_docs = file_loader(r"Data/")

In [5]:
def chunking_data(data):
  split_data = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 50)
  chunk_data = split_data.split_documents(data)
  return chunk_data

In [6]:
chunk_data = chunking_data(extracted_docs)
len(chunk_data)

2120

In [7]:
chunk_data[40].page_content

'backs (including the Tensor Board callback). • Chapter 11 (many changes)\n— Introduced self-normalizing nets, the SELU activation function and Alpha\nDropout. — Introduced self-supervised learning. — Added Nadam optimiza-\ntion. — Added Monte-Carlo Dropout. — Added a note about the risks of\nadaptive optimization methods. — Updated the practical guidelines. • Chap-\nter 12 – completely rewritten chapter, including: — A tour of TensorFlow 2 —'

In [8]:
def get_embedding():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [9]:
embedding = get_embedding()

C:\Users\SAMANWAYA\AppData\Local\Temp\ipykernel_2912\2649802877.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [10]:
docs = FAISS.from_documents(documents=chunk_data, embedding=embedding )
docs

In [13]:
retriver = docs.as_retriever(search_type="similarity", search_kwargs={"k": 3})
output = retriver.invoke("What is Supervised Machine Learning")
output

[Document(id='7a39d922-0c47-441b-b27c-5311cb2c619b', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 'page': 11, 'page_label': '12'}, page_content='Figure 1-5. A labeled training set for supervised learning (e.g., spam classifica-\ntion) 8 | Chapter 1: The Machine Learning Landscape\nA typical supervised learning task is classification. The spam filter is a good\nexample of this: it is trained with many example emails along with their class\n(spam or ham), and it must learn how to classify new emails. Another typical\ntask is to predict a target numeric value, such as the price of a car, given a set of'),
 Document(id='bd809a1c-a5f5-427a-b985-c8fd462376f5', metadata={'producer': 'xdvipdfmx (20211117)', 'creator': 'LaTeX via pandoc', 'creationdate': '2025-01-23T13:03:17+00:00', 'source': 'Data\\Hands-On-Machine-Learning.pdf', 'total_pages': 290, 

In [14]:
import os
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

In [16]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate

In [18]:
system_prompt = (
"You are an expert Data Scientist assistat of qestion-answering tasks."
"Use the following pieces of retrieved context to answer "
"the question. If you don't find any related context then say that you "
"don't know. Do not give any halusinating answer of this. Use the three sentece maximum and keep the "
"answer concise."
"\n\n"
"{context}"
 )

chat_prompt = ChatPromptTemplate.from_messages([
  ("system", system_prompt),
  ("user", "{input}" )]
)

In [19]:
stuff_chain =create_stuff_documents_chain(llm, chat_prompt)
retriver_chain = create_retrieval_chain(retriver, stuff_chain)
question = "What is Machine Learnig?"
response_dict = retriver_chain.invoke({"input" : question})
# response = response_dict["answer"] if isinstance(response_dict, dict) else str(response_dict)
response = response_dict['answer']
response

'Machine Learning is a field of artificial intelligence that focuses on developing algorithms and techniques that allow computers to learn from and make predictions or decisions based on data. It encompasses a variety of methods, from simple linear regression to advanced deep learning techniques. The goal is to create programs capable of improving their performance as they are exposed to more data.'

In [20]:
question = "What is Tensorflow?"
response_dict = retriver_chain.invoke({"input" : question})
# response = response_dict["answer"] if isinstance(response_dict, dict) else str(response_dict)
response = response_dict['answer']
response

"TensorFlow is an open-source machine learning framework that allows for various applications such as natural language processing, recommender systems, and time series forecasting. Its core is similar to NumPy but with added GPU support and capabilities for distributed computing. TensorFlow's API revolves around tensors, which are multidimensional arrays."